# Householder QR

Q is represented by reflectors. Factorization and orthogonality are checked with three probe vectors, without forming Q.

In [16]:
import time
from pathlib import Path
import numpy as np
import torch
from scipy.io import loadmat
from scipy.sparse import csr_matrix, issparse

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
print("PyTorch:", torch.__version__, "Device:", device)


def data_path(name, path=None):
    if path is not None:
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(p)
        return p
    matches = sorted(Path("/kaggle/input").rglob(f"{name}.mat"))
    if not matches:
        p = Path("data") / f"{name}.mat"
        if p.is_file():
            return p
    if len(matches) != 1:
        raise FileNotFoundError(f"Attach {name}.mat or set MAT_PATH; matches: {matches}")
    return matches[0]


def tensor(A, sparse=False):
    if sparse:
        C = csr_matrix(A, dtype=np.float64)
        C.sum_duplicates()
        C.sort_indices()
        return torch.sparse_csr_tensor(torch.as_tensor(C.indptr, dtype=torch.int64, device=device),
                                       torch.as_tensor(C.indices, dtype=torch.int64, device=device),
                                       torch.as_tensor(C.data, dtype=dtype, device=device),
                                       size=C.shape, dtype=dtype, device=device)
    return torch.tensor(A.toarray() if issparse(A) else A, dtype=dtype, device=device)


def mv(A, x):
    return (torch.sparse.mm(A, x[:, None])[:, 0] if A.layout == torch.sparse_csr else A @ x)


def timed(fn, *args):
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    result = fn(*args)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return result, time.perf_counter() - start


def backward_substitution(U, b):
    x = torch.zeros_like(b)
    for i in range(len(b)-1, -1, -1):
        if U[i, i] == 0:
            raise ValueError("Zero pivot")
        x[i] = (b[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x


def lu_solve(L, U, b, p=None):
    c = b.clone() if p is None else b[p].clone()
    for i in range(len(c)):
        c[i] = (c[i] - L[i, :i] @ c[:i]) / L[i, i]
    return backward_substitution(U, c)


def relative_residual(A, x, b):
    return (torch.linalg.vector_norm(mv(A, x)-b) / torch.linalg.vector_norm(b).clamp_min(torch.finfo(dtype).tiny)).item()


def iteration_info(A, x, b, k, tol):
    r = relative_residual(A, x, b)
    return {"iterations": k, "residual": r, "converged": np.isfinite(r) and r <= tol}


PyTorch: 2.11.0+cu130 Device: cuda


In [17]:
@torch.no_grad()
def householder_qr(A):
    R = A.clone()
    m, n = R.shape
    V = torch.zeros((m, min(m, n)), dtype=dtype, device=device)
    for k in range(min(m, n)):
        v = R[k:, k].clone()
        s = torch.linalg.vector_norm(v)
        if s == 0:
            continue
        v[0] += s if v[0] >= 0 else -s
        v /= torch.linalg.vector_norm(v)
        V[k:, k] = v
        R[k:, k:] -= 2 * torch.outer(v, v @ R[k:, k:])
        R[k+1:, k] = 0
    return V, R


@torch.no_grad()
def apply_q(V, B, transpose=False):
    C = B.clone()
    n = V.shape[1]
    for t in range(n):
        k = t if transpose else n-1-t
        v = V[k:, k]
        C[k:] -= 2 * torch.outer(v, v @ C[k:])
    return C


## 3×3


In [18]:
A = torch.tensor([[4., 1., 0.], [1., 3., 1.], [0., 1., 2.]], dtype=dtype, device=device)
x_true = torch.ones(3, dtype=dtype, device=device)
b = A @ x_true
Z = torch.randn((A.shape[1], 3), dtype=dtype, device=device)
_ = householder_qr(A)
result, seconds = timed(householder_qr, A)
print("Seconds:", seconds)
V, R = result
print("Factorization probe error:", float(torch.linalg.vector_norm(A @ Z - apply_q(V, R @ Z)) / torch.linalg.vector_norm(A @ Z)))
Y = torch.randn((A.shape[0], 3), dtype=dtype, device=device)
print("Orthogonality probe error:", float(torch.linalg.vector_norm(apply_q(V, apply_q(V, Y), transpose=True)-Y) / torch.linalg.vector_norm(Y)))
assert float(torch.linalg.vector_norm(A @ Z - apply_q(V, R @ Z)) / torch.linalg.vector_norm(A @ Z)) < 1e-12


Seconds: 0.0011356410000189499
Factorization probe error: 9.043409119145237e-16
Orthogonality probe error: 9.19604019580316e-16


## SuiteSparse: bcsstk17

Attach only bcsstk17.mat (10,974 × 10,974) using Add Input, or set MAT_PATH.

Dense float64 factorization costs O(n³). Each dense matrix needs about 0.90 GiB; the algorithm also needs working copies and factors.


In [19]:
MATRIX = "bcsstk17"
MAT_PATH = None
A_raw = loadmat(data_path(MATRIX, MAT_PATH))["Problem"][0, 0]["A"].astype(np.float64)
A = tensor(A_raw, sparse=False)
print("Matrix:", MATRIX, "shape:", A.shape)
Z = torch.randn((A.shape[1], 3), dtype=dtype, device=device)


Matrix: bcsstk17 shape: torch.Size([10974, 10974])


In [20]:
result, seconds = timed(householder_qr, A)
print("Seconds:", seconds)
V, R = result
print("Factorization probe error:", float(torch.linalg.vector_norm(A @ Z - apply_q(V, R @ Z)) / torch.linalg.vector_norm(A @ Z)))
Y = torch.randn((A.shape[0], 3), dtype=dtype, device=device)
print("Orthogonality probe error:", float(torch.linalg.vector_norm(apply_q(V, apply_q(V, Y), transpose=True)-Y) / torch.linalg.vector_norm(Y)))


Seconds: 126.98544045800008
Factorization probe error: 1.0801421547318286e-15
Orthogonality probe error: 9.642768498845312e-16
